In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00


In [2]:
import os
import shutil
import pandas as pd
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
import warnings

warnings.filterwarnings('ignore')

# --- 1. Paths (Kept exactly as per your working structure) ---
raw_dataset_dir = '/kaggle/input/datasets/hades199/vr-final-project-dataset/vr-final-project-dataset'

bbox_file = os.path.join(raw_dataset_dir, 'Anno/list_bbox_inshop.txt')
partition_file = os.path.join(raw_dataset_dir, 'Eval/list_eval_partition.txt')

yolo_base_dir = '/kaggle/working/yolo_dataset'
cropped_out_dir = '/kaggle/working/yolo-cropped-images'

# --- 2. Create YOLO Directory Structure ---
for split in ['train', 'val']:
    os.makedirs(os.path.join(yolo_base_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(yolo_base_dir, 'labels', split), exist_ok=True)

# --- 3. Load Metadata & 🆕 MAP CLASSES DIRECTLY ---
print("Loading annotations and partitions...")
df_bbox = pd.read_csv(bbox_file, sep=r'\s+', skiprows=1)
df_part = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)

# Merge together
df_merged = pd.merge(df_bbox, df_part, on='image_name')

# 🚨 THE FIX: Use clothes_type directly from bbox_file
# 1 (Upper) -> 0, 2 (Lower) -> 1, 3 (Full) -> 2
df_merged['yolo_class'] = df_merged['clothes_type'] - 1

df_train = df_merged[df_merged['evaluation_status'] == 'train']
df_val   = df_merged[df_merged['evaluation_status'] == 'query']

# ✅ Your original path discovery logic
def find_image_root(raw_dataset_dir, sample_rel_path):
    candidates = [
        raw_dataset_dir,
        os.path.join(raw_dataset_dir, 'img'),
        os.path.join(raw_dataset_dir, 'images'),
    ]
    try:
        for entry in os.scandir(raw_dataset_dir):
            if entry.is_dir():
                candidates.append(entry.path)
    except Exception:
        pass

    for root in candidates:
        full = os.path.join(root, sample_rel_path)
        stripped = os.path.join(root, sample_rel_path.replace('img/', ''))
        if os.path.exists(full):
            return root, sample_rel_path
        if os.path.exists(stripped):
            return root, sample_rel_path.replace('img/', '')

    raise FileNotFoundError("Could not find images. Check directory structure.")

sample_path = df_merged['image_name'].iloc[0]
img_base_dir, path_prefix_strip = find_image_root(raw_dataset_dir, sample_path)
strip_img_prefix = (path_prefix_strip != sample_path)

# --- 4. DeepFashion to YOLO Converter ---
def resolve_path(img_rel_path, img_base_dir, strip_img_prefix):
    rel = img_rel_path.replace('img/', '') if strip_img_prefix else img_rel_path
    return os.path.join(img_base_dir, rel)

def convert_to_yolo(df, split_name):
    print(f"\nPreparing {split_name} data with 3 classes...")
    successful_copies = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_rel_path = row['image_name']
        img_full_path = resolve_path(img_rel_path, img_base_dir, strip_img_prefix)

        if not os.path.exists(img_full_path):
            continue

        try:
            with Image.open(img_full_path) as img:
                w_img, h_img = img.size
        except:
            continue

        x1, y1, x2, y2 = row['x_1'], row['y_1'], row['x_2'], row['y_2']
        
        # 🚨 Use the new class we mapped
        yolo_class = int(row['yolo_class'])

        w_box    = max(min((x2 - x1) / w_img, 1.0), 0.0)
        h_box    = max(min((y2 - y1) / h_img, 1.0), 0.0)
        x_center = max(min((x1 + (x2 - x1) / 2) / w_img, 1.0), 0.0)
        y_center = max(min((y1 + (y2 - y1) / 2) / h_img, 1.0), 0.0)

        safe_name = img_rel_path.replace('/', '_')
        shutil.copy(img_full_path, os.path.join(yolo_base_dir, 'images', split_name, safe_name))

        label_name = safe_name.rsplit('.', 1)[0] + '.txt'
        with open(os.path.join(yolo_base_dir, 'labels', split_name, label_name), 'w') as f:
            f.write(f"{yolo_class} {x_center:.6f} {y_center:.6f} {w_box:.6f} {h_box:.6f}\n")

        successful_copies += 1

convert_to_yolo(df_train, 'train')
convert_to_yolo(df_val,   'val')

# --- 6. Generate data.yaml with Correct Names ---
yaml_content = f"""path: {yolo_base_dir}
train: images/train
val: images/val

names:
  0: Upper Body
  1: Lower Body
  2: Full Body
"""
with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

# --- 7. Train YOLOv8 ---
model = YOLO('yolov8n.pt')
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=15,
    imgsz=640,
    batch=32,
    device=0,
    project='/kaggle/working/',
    name='deepfashion_yolo',
    save=True
)

# --- 8. Smart Crop entire dataset ---
best_model = YOLO('/kaggle/working/deepfashion_yolo/weights/best.pt')

for _, row in tqdm(df_merged.iterrows(), total=len(df_merged), desc="Smart Cropping"):
    img_full_path = resolve_path(row['image_name'], img_base_dir, strip_img_prefix)
    if not os.path.exists(img_full_path):
        continue

    dest_path = os.path.join(cropped_out_dir, row['image_name'])
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    img = Image.open(img_full_path).convert('RGB')
    inference_results = best_model.predict(img, verbose=False, device=0)
    
    target_cls = int(row['yolo_class'])
    best_box = None

    if len(inference_results[0].boxes) > 0:
        for box in inference_results[0].boxes:
            if int(box.cls[0].cpu().numpy()) == target_cls:
                best_box = box.xyxy[0].cpu().numpy()
                break
        
        if best_box is None:
            best_box = inference_results[0].boxes[0].xyxy[0].cpu().numpy()
            
        img.crop(tuple(map(int, best_box))).save(dest_path)
    else:
        img.save(dest_path)

# --- 9. Zip ---
shutil.make_archive('/kaggle/working/custom_yolo_crops', 'zip', cropped_out_dir)
print("✅ Done! Multi-class YOLO trained and Gallery cropped.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading annotations and partitions...

Preparing train data with 3 classes...


100%|██████████| 25882/25882 [03:14<00:00, 132.86it/s]



Preparing val data with 3 classes...


100%|██████████| 14218/14218 [01:54<00:00, 124.63it/s]


Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=deepfashion_yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

Smart Cropping: 100%|██████████| 52712/52712 [12:43<00:00, 69.05it/s]


✅ Done! Multi-class YOLO trained and Gallery cropped.
